# Tutorial: Plan 004 Motivation — Safe Managed Operations

**Audience:** contributors designing Archiver's first filesystem-mutating workflow.  
**Prerequisites:** Plans 002–003, especially reconciliation, current-state queries, and duplicate inspection.

By the end, you will be able to:

- explain why duplicate discovery alone cannot authorize deletion;
- distinguish observed storage from explicitly managed storage;
- build a prospective, reviewable quarantine plan with an explicit keeper;
- validate catalog and filesystem preconditions before any mutation;
- explain why quarantine and an operation log should precede permanent deletion.

## Outline

1. Create a synthetic catalog containing duplicate content.
2. Inspect duplicates with the current Plan 003 API.
3. Model the authority boundary proposed for Plan 004.
4. Build a descriptive quarantine plan without executing it.
5. Detect a file that changes after planning.
6. Review the recoverable operation lifecycle and common pitfalls.

> This notebook demonstrates motivation and prospective usage. Plan 004 is not implemented here, and no file is moved or deleted.

## Why Plan 004 is needed

A duplicate group says that several observed paths had the same bytes during the current successful scan. It does **not** say:

- which pathname is authoritative;
- whether Archiver may modify that location;
- whether the filesystem still matches the catalog;
- whether losing metadata or a particular directory layout is acceptable;
- whether an interrupted multi-file operation can be recovered.

Plan 004 should therefore establish authority and recoverable operation semantics before adding deduplication. The safest initial physical action is a verified move into quarantine, not permanent deletion.

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path, PurePosixPath
from tempfile import TemporaryDirectory
from typing import Literal

from archiver import Catalog, ContentId, FileObservation
from archiver.hashing import hash_file_stably

workspace = TemporaryDirectory(prefix="archiver-plan004-")
workspace_path = Path(workspace.name)
root = workspace_path / "candidate-root"
root.mkdir()

files = {
    "photos/original/photo.jpg": b"synthetic photo bytes",
    "imports/phone/photo-copy.jpg": b"synthetic photo bytes",
    "backup/photo-backup.jpg": b"synthetic photo bytes",
    "notes/readme.txt": b"unique notes",
}
for relative_path, content in files.items():
    path = root / relative_path
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_bytes(content)

catalog = Catalog.create(workspace_path / "catalog.sqlite")
summary = catalog.scan_directory(root)
print(f"Cataloged {summary.files_observed} files; duplicate groups: {summary.duplicate_content_group_count}")

## 1. Inspect duplicates with today's API

Plan 003 provides bounded, descriptive inspection. The displayed members are observations tied to one current scan; no member is a keeper candidate by default.

In [ ]:
duplicate_search = catalog.search_duplicate_groups(root, group_limit=10, member_limit=10)

for group in duplicate_search.groups:
    print(
        f"{group.content_id.digest[:12]}… | {group.size_bytes} bytes | "
        f"{group.file_instance_count} copies | {group.potential_redundant_bytes} potentially redundant bytes"
    )
    for member in group.members:
        print(f"  {member.relative_path.as_posix()}")

The result establishes content equality, but all three paths may carry different human meaning. Lexicographic order, modification time, or directory depth would be arbitrary keeper policies unless a user explicitly chooses them.

## 2. Make mutation authority explicit

The current catalog is observational. A future Plan 004 location policy could distinguish roots Archiver may only observe from roots explicitly placed under management.

In [ ]:
Authority = Literal["observed", "managed"]


@dataclass(frozen=True, slots=True)
class ProposedLocationPolicy:
    root: Path
    authority: Authority


def require_managed(policy: ProposedLocationPolicy) -> None:
    if policy.authority != "managed":
        raise PermissionError("the location is observational; mutation is not authorized")


observed_policy = ProposedLocationPolicy(root=root, authority="observed")
managed_policy = ProposedLocationPolicy(root=root, authority="managed")

for policy in (observed_policy, managed_policy):
    try:
        require_managed(policy)
    except PermissionError as error:
        print(f"{policy.authority}: rejected — {error}")
    else:
        print(f"{policy.authority}: eligible for operation planning")

This local dataclass is only a design sketch. In the real system, managed authority must be persisted explicitly and changing it should be an intentional command—not inferred from the existence of a catalog database.

## 3. Build a reviewable quarantine plan

A useful operation plan records:

- the managed root and catalog baseline;
- one keeper selected by the user or an explicit policy;
- the expected identity and metadata of every affected path;
- deterministic quarantine destinations;
- an operation identifier that can tie actions to a future operation log.

The bounded inspection result is for display. Planning retrieves the complete group by content identity before constructing actions.

In [ ]:
@dataclass(frozen=True, slots=True)
class FilePrecondition:
    relative_path: PurePosixPath
    content_id: ContentId
    size_bytes: int
    mtime_ns: int


@dataclass(frozen=True, slots=True)
class QuarantineAction:
    source: FilePrecondition
    quarantine_relative_path: PurePosixPath


@dataclass(frozen=True, slots=True)
class QuarantinePlan:
    operation_id: str
    root: Path
    baseline_scan_id: int
    keeper: FilePrecondition
    actions: tuple[QuarantineAction, ...]


def as_precondition(observation: FileObservation) -> FilePrecondition:
    return FilePrecondition(
        relative_path=observation.relative_path,
        content_id=observation.content_id,
        size_bytes=observation.size_bytes,
        mtime_ns=observation.mtime_ns,
    )


def build_quarantine_plan(
    policy: ProposedLocationPolicy,
    members: tuple[FileObservation, ...],
    keeper_path: PurePosixPath,
    baseline_scan_id: int,
    operation_id: str,
) -> QuarantinePlan:
    require_managed(policy)
    if len(members) < 2:
        raise ValueError("resolution requires a complete duplicate group")
    by_path = {member.relative_path: member for member in members}
    if keeper_path not in by_path:
        raise ValueError("the keeper must be a member of the duplicate group")
    keeper = as_precondition(by_path[keeper_path])
    actions = tuple(
        QuarantineAction(
            source=as_precondition(member),
            quarantine_relative_path=PurePosixPath(".archiver/quarantine") / operation_id / member.relative_path,
        )
        for member in members
        if member.relative_path != keeper_path
    )
    return QuarantinePlan(operation_id, policy.root, baseline_scan_id, keeper, actions)

In [ ]:
displayed_group = duplicate_search.groups[0]
complete_members = tuple(catalog.find_by_content(root, displayed_group.content_id))
current_scan = catalog.current_scan(root)
assert current_scan is not None

keeper_path = PurePosixPath("photos/original/photo.jpg")  # explicit tutorial choice
plan = build_quarantine_plan(
    managed_policy,
    complete_members,
    keeper_path,
    baseline_scan_id=current_scan.id,
    operation_id="demo-0001",
)

print(f"Keep: {plan.keeper.relative_path.as_posix()}")
for action in plan.actions:
    print(f"Quarantine: {action.source.relative_path.as_posix()} -> {action.quarantine_relative_path.as_posix()}")

The plan is reviewable and deterministic, but it still grants no permission to execute itself. A future executor must load persisted managed authority, record the operation, and revalidate every precondition immediately before moving files.

## Exercise: choose a different keeper

Change `alternate_keeper` to another duplicate member. Predict which two paths become quarantine actions before running the cell. Notice that content identity stays the same while the path policy changes.

In [ ]:
# Answer scaffold: edit this path and rerun the cell.
alternate_keeper = PurePosixPath("backup/photo-backup.jpg")
alternate_plan = build_quarantine_plan(
    managed_policy,
    complete_members,
    alternate_keeper,
    baseline_scan_id=current_scan.id,
    operation_id="demo-0002",
)
print("Alternate actions:")
for action in alternate_plan.actions:
    print(f"  {action.source.relative_path.as_posix()}")

## 4. Revalidate catalog and filesystem preconditions

A catalog baseline detects another applied refresh, but the filesystem can change without a refresh. Plan 004 therefore needs both checks: compare the current scan ID and re-read the keeper and every action source.

In [ ]:
def validate_plan(catalog: Catalog, plan: QuarantinePlan) -> tuple[str, ...]:
    errors: list[str] = []
    scan = catalog.current_scan(plan.root)
    if scan is None or scan.id != plan.baseline_scan_id:
        errors.append("catalog baseline changed")

    checks = (plan.keeper, *(action.source for action in plan.actions))
    for expected in checks:
        path = plan.root / Path(expected.relative_path.as_posix())
        try:
            content_id, size_bytes, mtime_ns = hash_file_stably(path)
        except OSError as error:
            errors.append(f"{expected.relative_path.as_posix()}: unavailable ({error})")
            continue
        if (content_id, size_bytes, mtime_ns) != (
            expected.content_id,
            expected.size_bytes,
            expected.mtime_ns,
        ):
            errors.append(f"{expected.relative_path.as_posix()}: no longer matches the plan")
    return tuple(errors)


print("Validation before change:", validate_plan(catalog, plan))

Now simulate an external edit after planning. This changes only a synthetic temporary file. The catalog baseline remains the same, so byte/metadata preconditions provide the additional protection.

In [ ]:
changed_source = plan.root / Path(plan.actions[0].source.relative_path.as_posix())
changed_source.write_bytes(b"changed after planning")

print("Catalog scan ID is still:", catalog.current_scan(root).id)
print("Validation after external change:")
for error in validate_plan(catalog, plan):
    print(f"  {error}")

## 5. Proposed recoverable lifecycle

A first Plan 004 implementation should stop at quarantine:

1. **Plan:** require managed authority, an explicit keeper, a current catalog baseline, and complete members.
2. **Record:** persist the operation and intended actions with status `planned`.
3. **Validate:** re-hash keeper and sources and ensure destinations are unused.
4. **Execute:** mark `running`, then move each redundant source into an operation-specific quarantine path.
5. **Record partial results:** never report success when only some moves completed.
6. **Refresh:** reconcile and apply a new catalog snapshot after successful moves.
7. **Recover:** support restoring quarantined paths while their original destinations are safe.
8. **Purge later:** permanent deletion should be a separate, explicit policy after a retention period.

The notebook intentionally does not execute the following previewed moves:

In [ ]:
for action in plan.actions:
    print(
        f"WOULD MOVE {action.source.relative_path.as_posix()} "
        f"TO {action.quarantine_relative_path.as_posix()}"
    )

## Pitfalls and extensions

- **Bounded display is not a complete operation input.** Resolve the selected content identity back to every current member before planning.
- **A path is not content identity.** Re-hash immediately before mutation.
- **A catalog is not authority.** Reject planning and execution for observational locations.
- **A baseline is not a filesystem lock.** Validate both the current scan and live files.
- **Quarantine is not atomic as a group.** Persist per-action status so interruption can be recovered.
- **Missing observations are not deletion instructions.** They already describe paths absent from the latest successful state.

Optional extension: add proposed `planned`, `running`, `completed`, `failed`, and `recovered` operation records, then model an undo command that checks destination conflicts before restoring each quarantined path.

In [ ]:
catalog.close()
workspace.cleanup()
print("Synthetic tutorial workspace removed.")